# **ETL of World Bank GDP Data**

## Objectives
* read in World Bank's GDP
* clean the data, apply the iso3 country code
* append to CO2 and population data

## Inputs
* data/processed/01b_df_owid_co2_pop_1970_2024.csv

## Outputs

* data/processed/01d_df_main_1970_2024.csv


---

# Set Up directories and Import Necessary Libraries

In [98]:
# System and OS related tasks
import sys
import os

# Add the project root to Python path
project_root = os.path.abspath('..')
sys.path.insert(0, project_root)

# path to directories
raw_dir = '../data/raw'
processed_dir = '../data/processed'

In [99]:
# Get the necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid') # sets a white background with grid lines 

## 1.0 Get the List of Countries from the CO2 and Population data

In [100]:
data_columns = [
    "country"
    , "country_code"
    , "year"
    , "co2_pc"    
    , "country_code_iso3"
    , "country_name_iso3"
    , "population"
]

column_dtypes = {
    "country": str
    , "country_code": str
    , "year": int
    , "co2_pc" : float
    , "country_code_iso3": str
    , "country_name_iso3": str
    , "population" : float
    }

In [108]:
file_name = "01b_df_owid_co2_pop_1970_2024.csv"

df_owid_co2_pop_1970_2024 = pd.read_csv(
    f"{processed_dir}/{file_name}",
    header=0,
    names=data_columns,
    dtype=column_dtypes
    )

print(f"Total number of rows: {len(df_owid_co2_pop_1970_2024)}")

Total number of rows: 11479


In [102]:
df_owid_co2_pop_1970_2024.head(3)

,country,country_code,year,co2_pc,country_code_iso3,country_name_iso3,population
0,Afghanistan,AFG,1970,0.147952,AFG,Afghanistan,11290134.0
1,Afghanistan,AFG,1971,0.163694,AFG,Afghanistan,11567672.0
2,Afghanistan,AFG,1972,0.129103,AFG,Afghanistan,11853698.0


In [103]:
country_list = sorted(list(set(df_owid_co2_pop_1970_2024["country_code_iso3"])))
country_list

['ABW',
 'AFG',
 'AGO',
 'AIA',
 'ALB',
 'AND',
 'ARE',
 'ARG',
 'ARM',
 'ATG',
 'AUS',
 'AUT',
 'AZE',
 'BDI',
 'BEL',
 'BEN',
 'BES',
 'BFA',
 'BGD',
 'BGR',
 'BHR',
 'BHS',
 'BIH',
 'BLR',
 'BLZ',
 'BMU',
 'BOL',
 'BRA',
 'BRB',
 'BRN',
 'BTN',
 'BWA',
 'CAF',
 'CAN',
 'CHE',
 'CHL',
 'CHN',
 'CIV',
 'CMR',
 'COD',
 'COG',
 'COK',
 'COL',
 'COM',
 'CPV',
 'CRI',
 'CUB',
 'CUW',
 'CYP',
 'CZE',
 'DEU',
 'DJI',
 'DMA',
 'DNK',
 'DOM',
 'DZA',
 'ECU',
 'EGY',
 'ERI',
 'ESP',
 'EST',
 'ETH',
 'FIN',
 'FJI',
 'FRA',
 'FRO',
 'FSM',
 'GAB',
 'GBR',
 'GEO',
 'GHA',
 'GIN',
 'GMB',
 'GNB',
 'GNQ',
 'GRC',
 'GRD',
 'GRL',
 'GTM',
 'GUY',
 'HKG',
 'HND',
 'HRV',
 'HTI',
 'HUN',
 'IDN',
 'IND',
 'IRL',
 'IRN',
 'IRQ',
 'ISL',
 'ISR',
 'ITA',
 'JAM',
 'JOR',
 'JPN',
 'KAZ',
 'KEN',
 'KGZ',
 'KHM',
 'KIR',
 'KNA',
 'KOR',
 'KWT',
 'LAO',
 'LBN',
 'LBR',
 'LBY',
 'LCA',
 'LIE',
 'LKA',
 'LSO',
 'LTU',
 'LUX',
 'LVA',
 'MAC',
 'MAR',
 'MDA',
 'MDG',
 'MDV',
 'MEX',
 'MHL',
 'MKD',
 'MLI',
 'MLT',


In [104]:
print(f"Total number of countries: {len(country_list)}")

Total number of countries: 214


## 2.0 World Bank's GDP Data (wbgapi package)

📌 The wbgapi is a package for accessing World Bank's data API. See [wbgapi package](https://pypi.org/project/wbgapi/).

In [105]:
import wbgapi as wb

In [106]:
# take a look at the GDP data available via this package
wb.series.info(q='GDP')

id,value
EG.GDP.PUSE.KO.PP,GDP per unit of energy use (PPP $ per kg of oil equivalent)
EG.GDP.PUSE.KO.PP.KD,GDP per unit of energy use (constant 2021 PPP $ per kg of oil equivalent)
EG.USE.COMM.GD.PP.KD,"Energy use (kg of oil equivalent) per $1,000 GDP (constant 2021 PPP)"
EN.GHG.CO2.RT.GDP.KD,Carbon intensity of GDP (kg CO2e per constant 2015 US$ of GDP)
EN.GHG.CO2.RT.GDP.PP.KD,Carbon intensity of GDP (kg CO2e per 2021 PPP $ of GDP)
NY.GDP.DEFL.KD.ZG,"Inflation, GDP deflator (annual %)"
NY.GDP.DEFL.KD.ZG.AD,"Inflation, GDP deflator: linked series (annual %)"
NY.GDP.DEFL.ZS,GDP deflator (base year varies by country)
NY.GDP.DEFL.ZS.AD,GDP deflator: linked series (base year varies by country)
NY.GDP.DISC.CN,Discrepancy in expenditure estimate of GDP (current LCU)


📌 `NY.GDP.PCAP.PP.KD` for `GDP per capita, PPP (constant 2021 international $)` is chosen for this project as this GDP measure:
* uses constant price (fixed to 2021 base year)
* removes inflation and exchange rate fluctuations


In [109]:
wb.series.info("NY.GDP.PCAP.PP.KD")

id,value
NY.GDP.PCAP.PP.KD,"GDP per capita, PPP (constant 2021 international $)"
,1 elements


In [110]:
# Get the GDP data for 25 years for the top tea and coffee producting countries
df_wb_gdp_wide = wb.data.DataFrame(
    "NY.GDP.PCAP.PP.KD"
    , country_list # country_ISO
    , time=range(1970,2025) # 2025 so that 2024 is included.
    , labels=True
).reset_index()

df_wb_gdp_wide.head(5)

,economy,Country,YR1970,YR1971,YR1972,YR1973,YR1974,YR1975,YR1976,YR1977,...,YR2015,YR2016,YR2017,YR2018,YR2019,YR2020,YR2021,YR2022,YR2023,YR2024
0,ZWE,Zimbabwe,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,5102.714323,5070.402170,5234.383866,5415.469764,4993.843839,4527.719881,4827.088694,5036.761361,5218.022665,5215.253011
1,ZMB,Zambia,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3576.925448,3598.171660,3612.505977,3646.959665,3591.564189,3391.595412,3503.034914,3585.123791,3673.484197,3708.069043
2,ZAF,South Africa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,14541.674861,14501.280782,14573.598084,14553.561567,14352.669871,13250.566659,13681.954814,13767.496759,13695.375569,13597.653850
3,YEM,"Yemen, Rep.",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,XKX,Kosovo,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,9824.991060,10434.888826,10856.290207,11188.224463,11774.117851,11137.502856,12362.421018,13022.462744,14240.203032,15716.292556


In [111]:
#Convert the wide format data to long
## using the economy and country columns to unpivot the wide data to long data such that each long for the values of economy-country-year
df_wb_gdp_long = pd.melt(
    df_wb_gdp_wide
    , id_vars=['economy', 'Country']
    , var_name="year"
    , value_name="gdp_constant_ppp_pc"
    )

df_wb_gdp_long.head(10)

,economy,Country,year,gdp_constant_ppp_pc
0,ZWE,Zimbabwe,YR1970,NaN
1,ZMB,Zambia,YR1970,NaN
2,ZAF,South Africa,YR1970,NaN
3,YEM,"Yemen, Rep.",YR1970,NaN
4,XKX,Kosovo,YR1970,NaN
5,WSM,Samoa,YR1970,NaN
6,VUT,Vanuatu,YR1970,NaN
7,VNM,Viet Nam,YR1970,NaN
8,VGB,British Virgin Islands,YR1970,NaN
9,VEN,"Venezuela, RB",YR1970,NaN


In [112]:
# remove the unnecessary "YR" from the year value and convert year into integar
df_wb_gdp_long["year"] = df_wb_gdp_long["year"].str.replace("YR","")

df_wb_gdp_long["year"] = df_wb_gdp_long["year"].astype(int)

df_wb_gdp_long.head(10)

,economy,Country,year,gdp_constant_ppp_pc
0,ZWE,Zimbabwe,1970,NaN
1,ZMB,Zambia,1970,NaN
2,ZAF,South Africa,1970,NaN
3,YEM,"Yemen, Rep.",1970,NaN
4,XKX,Kosovo,1970,NaN
5,WSM,Samoa,1970,NaN
6,VUT,Vanuatu,1970,NaN
7,VNM,Viet Nam,1970,NaN
8,VGB,British Virgin Islands,1970,NaN
9,VEN,"Venezuela, RB",1970,NaN


In [113]:
import country_converter as coco

cc = coco.CountryConverter()

df_wb_gdp_long["country_code_iso3"] = cc.pandas_convert(
    series=df_wb_gdp_long["economy"]
    ,to="ISO3"
)

df_wb_gdp_long.head(3)

,economy,Country,year,gdp_constant_ppp_pc,country_code_iso3
0,ZWE,Zimbabwe,1970,NaN,ZWE
1,ZMB,Zambia,1970,NaN,ZMB
2,ZAF,South Africa,1970,NaN,ZAF


In [114]:
df_wb_gdp_long = df_wb_gdp_long[["country_code_iso3", "year", "gdp_constant_ppp_pc"]]

df_wb_gdp_long.head(3)

,country_code_iso3,year,gdp_constant_ppp_pc
0,ZWE,1970,NaN
1,ZMB,1970,NaN
2,ZAF,1970,NaN


In [ ]:
df_wb_gdp_long = df_wb_gdp_long.sort_values(["country_code_iso3", "year", "gdp_ppp_pc"]).reset_index(drop=True)

df_wb_gdp_long.head(10)

,country_code_iso3,year,gdp_ppp_pc
0,ABW,1970,NaN
1,ABW,1971,NaN
2,ABW,1972,NaN
3,ABW,1973,NaN
4,ABW,1974,NaN
5,ABW,1975,NaN
6,ABW,1976,NaN
7,ABW,1977,NaN
8,ABW,1978,NaN
9,ABW,1979,NaN


In [115]:
#check for duplcates country-year rows
country_year_counts = df_wb_gdp_long.groupby(
    ["country_code_iso3", "year"]
).size()

print(f"Number of duplicated country-year: {len(country_year_counts[country_year_counts > 1])}")

Number of duplicated country-year: 0


In [ ]:
# Count countries with non-NaN GDP per year
countries_per_year = df_wb_gdp_long.groupby("year")["gdp_constant_ppp_pc"].count().reset_index()
countries_per_year.columns = ["year", "countries_with_gdp"]

print("="*80)
print("COUNTRIES WITH GDP DATA PER YEAR:")
print("="*80)
print(countries_per_year.to_string(index=False))

COUNTRIES WITH GDP DATA PER YEAR:
 year  countries_with_gdp
 1970                   0
 1971                   0
 1972                   0
 1973                   0
 1974                   0
 1975                   0
 1976                   0
 1977                   0
 1978                   0
 1979                   0
 1980                   0
 1981                   0
 1982                   0
 1983                   0
 1984                   0
 1985                   0
 1986                   0
 1987                   0
 1988                   0
 1989                   0
 1990                 184
 1991                 185
 1992                 185
 1993                 185
 1994                 186
 1995                 187
 1996                 187
 1997                 188
 1998                 188
 1999                 188
 2000                 190
 2001                 190
 2002                 190
 2003                 190
 2004                 190
 2005                 190
 200

In [116]:
df_wb_gdp_long_1990_2024 = df_wb_gdp_long[df_wb_gdp_long["year"] >= 1990]

print(f"Min/Max Years df_wb_gdp_long_1990_2024: {df_wb_gdp_long_1990_2024["year"].min()}/{df_wb_gdp_long_1990_2024["year"].max()} ")

Min/Max Years df_wb_gdp_long_1990_2024: 1990/2024 


---

# 3.0 Merge GDP with CO2 and Population Data

📌 Since the GDP data only starts from 1990. We will also only take the CO2 and population data from 1990

In [117]:
df_owid_co2_pop_1970_2024.head(5)

,country,country_code,year,co2_pc,country_code_iso3,country_name_iso3,population
0,Afghanistan,AFG,1970,0.147952,AFG,Afghanistan,11290134.0
1,Afghanistan,AFG,1971,0.163694,AFG,Afghanistan,11567672.0
2,Afghanistan,AFG,1972,0.129103,AFG,Afghanistan,11853698.0
3,Afghanistan,AFG,1973,0.134517,AFG,Afghanistan,12158000.0
4,Afghanistan,AFG,1974,0.153431,AFG,Afghanistan,12469127.0


In [118]:
df_owid_co2_pop_1990_2024 = df_owid_co2_pop_1970_2024[df_owid_co2_pop_1970_2024["year"] >= 1990]

print(f"Min/Max Years df_owid_co2_pop_1990_2024: {df_owid_co2_pop_1990_2024["year"].min()}/{df_owid_co2_pop_1990_2024["year"].max()} ")

Min/Max Years df_owid_co2_pop_1990_2024: 1990/2024 


In [119]:
df_co2_pop_gdp_1990_2024 = df_owid_co2_pop_1990_2024.merge(
    df_wb_gdp_long_1990_2024[["country_code_iso3", "year", "gdp_constant_ppp_pc"]]
    , on=["country_code_iso3", "year"]
    , how="left"
)

df_co2_pop_gdp_1990_2024.head(5)

,country,country_code,year,co2_pc,country_code_iso3,country_name_iso3,population,gdp_constant_ppp_pc
0,Afghanistan,AFG,1990,0.168054,AFG,Afghanistan,12045664.0,NaN
1,Afghanistan,AFG,1991,0.156411,AFG,Afghanistan,12238879.0,NaN
2,Afghanistan,AFG,1992,0.111609,AFG,Afghanistan,13278983.0,NaN
3,Afghanistan,AFG,1993,0.099506,AFG,Afghanistan,14943175.0,NaN
4,Afghanistan,AFG,1994,0.089462,AFG,Afghanistan,16250800.0,NaN


In [120]:
print(f"Total number of rows in df_co2_pop_gdp_1990_2024: {len(df_co2_pop_gdp_1990_2024)}")
print(f"Rows with GDP data: {df_co2_pop_gdp_1990_2024['gdp_constant_ppp_pc'].notna().sum()}")
print(f"Rows missing GDP: {df_co2_pop_gdp_1990_2024['gdp_constant_ppp_pc'].isna().sum()}")

Total number of rows in df_co2_pop_gdp_1990_2024: 7473
Rows with GDP data: 6675
Rows missing GDP: 798


In [121]:
missing_gdp_rows = df_co2_pop_gdp_1990_2024[df_co2_pop_gdp_1990_2024["gdp_constant_ppp_pc"].isna()].copy()

missing_gdp_rows["country_name_iso3"].value_counts()

country_name_iso3
Anguilla                             35
Bonaire, Saint Eustatius and Saba    35
British Virgin Islands               35
Cook Islands                         35
Cuba                                 35
French Polynesia                     35
Liechtenstein                        35
Montserrat                           35
New Caledonia                        35
Niue                                 35
North Korea                          35
St. Helena                           35
St. Pierre and Miquelon              35
South Sudan                          35
Taiwan                               35
Venezuela                            35
Wallis and Futuna Islands            35
Yemen                                35
Eritrea                              31
Djibouti                             23
Sint Maarten                         19
Faroe Islands                        18
Turks and Caicos Islands             17
Kosovo                               14
Afghanistan           

In [124]:
desired_columns = [
    "country_code_iso3"
    , "country_name_iso3"
    , "year"
    , "co2_pc"
    , "population"
    , "gdp_constant_ppp_pc"
]

df_co2_pop_gdp_1990_2024 = df_co2_pop_gdp_1990_2024[desired_columns]

df_co2_pop_gdp_1990_2024.head(10)

,country_code_iso3,country_name_iso3,year,co2_pc,population,gdp_constant_ppp_pc
0,AFG,Afghanistan,1990,0.168054,12045664.0,NaN
1,AFG,Afghanistan,1991,0.156411,12238879.0,NaN
2,AFG,Afghanistan,1992,0.111609,13278983.0,NaN
3,AFG,Afghanistan,1993,0.099506,14943175.0,NaN
4,AFG,Afghanistan,1994,0.089462,16250800.0,NaN
5,AFG,Afghanistan,1995,0.083051,17065837.0,NaN
6,AFG,Afghanistan,1996,0.077131,17763266.0,NaN
7,AFG,Afghanistan,1997,0.070678,18452100.0,NaN
8,AFG,Afghanistan,1998,0.066728,19159996.0,NaN
9,AFG,Afghanistan,1999,0.054890,19887792.0,NaN


---

# 4.0 Export to CSV

In [125]:
df_co2_pop_gdp_1990_2024.to_csv(f'{processed_dir}/01d_df_co2_pop_gdp_1990_2024.csv', index=False)
print(f"Exported: {processed_dir}/01d_df_co2_pop_gdp_1990_2024.csv")

Exported: ../data/processed/01d_df_co2_pop_gdp_1990_2024.csv


---